# DQN CartPole — Practice Notebook

This notebook is the hands-on companion to the **SnapCode DQN-101** course.  
You will implement every concept from Ch1–Ch6 in real Python code.

| Cell | Content | Course connection |
|---|---|---|
| 1 | Install & import | — |
| 2 | Neural Network | Ch4 · DQN pipeline |
| 3 | Replay Buffer | Ch5 · Replay Buffer |
| 4 | DQN Agent | Ch5 + Ch6 |
| 5 | Training Loop | Ch6 · Full loop |
| 6 | Results & plots | Ch6 · Learning curve |

---
**Goal**: after 300 episodes, the agent should balance the pole for 200 steps (CartPole-v1 max).

In [ ]:
# ── Cell 1: Install & Import ───────────────────────────────────────────────────

# Install required libraries
# -q = quiet mode (suppresses verbose output)
!pip install gymnasium matplotlib torch -q

# gymnasium — the CartPole physics simulator (Ch1 · RL environment)
# env = gym.make('CartPole-v1') creates the simulation
import gymnasium as gym

# torch — core tensor library (GPU-accelerated numpy)
import torch

# torch.nn — neural network layers: Linear, ReLU, etc. (Ch4 · DQN pipeline)
import torch.nn as nn

# torch.optim — weight update algorithms
# Adam: adaptive learning rate optimizer, standard choice for DQN
import torch.optim as optim

# numpy — array operations, used to convert states into tensors
import numpy as np

# matplotlib — plotting the learning curve (Ch6 · Atari learning curve)
import matplotlib.pyplot as plt

# deque — double-ended queue used for the Replay Buffer (Ch5)
# deque(maxlen=N) automatically drops the oldest entry when full
from collections import deque

# random — used for ε-greedy action selection (Ch6 · exploration)
# random.random() < ε  →  explore (random action)
import random


# ── Verify setup ───────────────────────────────────────────────────────────────
print('Setup complete')
print(f'  PyTorch : {torch.__version__}')

# Check if a GPU is available
# CartPole → CPU is enough (state = 4 numbers, tiny network)
# Atari    → GPU required (state = 84×84×4 pixels, large CNN)
print(f'  Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (fine for CartPole)"}')

## Step 1 · Neural Network
**Ch4 · DQN pipeline**

CartPole state is 4 numbers — no pixels, so we use fully-connected layers instead of CNN.

```
Input (4)  →  Hidden (128)  →  Hidden (128)  →  Output (2)
[x, ẋ, θ, θ̇]                                   [Q(LEFT), Q(RIGHT)]

x  = cart position
ẋ  = cart velocity
θ  = pole angle
θ̇  = pole angular velocity
```

Both **Main Network** and **Target Network** share this exact architecture.  
The only difference is *when* their weights are updated.

In [ ]:
# ── Cell 2: Neural Network ─────────────────────────────────────────────────────
class DQNNetwork(nn.Module):
    def __init__(self, state_size=4, action_size=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_size)  # one Q value per action
        )

    def forward(self, x):
        return self.net(x)


# Quick test
net = DQNNetwork()
# CartPole state order: [x, x_dot, theta, theta_dot]
#   index 0: cart position
#   index 1: cart velocity
#   index 2: pole angle
#   index 3: pole angular velocity
test_state = torch.tensor([0.02, 0.01, -0.03, 0.02])  # [x, ẋ, θ, θ̇]
q_values = net(test_state)

print('Network test')
print(f'  Input state : {test_state.tolist()}  → [x, ẋ, θ, θ̇]')
print(f'  Q(LEFT)  = {q_values[0].item():.4f}')
print(f'  Q(RIGHT) = {q_values[1].item():.4f}')
print(f'  → action : {"LEFT" if q_values.argmax() == 0 else "RIGHT"} (random weights — meaningless for now)')
print(f'\n  Total parameters: {sum(p.numel() for p in net.parameters()):,}')

In [ ]:
# ── Inspect A: What does the network actually receive and output? ──────────────
# A real CartPole starting state — 4 float values
state_inspect = np.array([0.04, -0.01, -0.02, 0.03])

print('── Input ──────────────────────────────────────────────────')
print(f'  shape  : {state_inspect.shape}   ← 4 floats per timestep')
print(f'  values : {state_inspect}')
print(f'           [x={state_inspect[0]:.4f}, ẋ={state_inspect[1]:.4f}, θ={state_inspect[2]:.4f}, θ̇={state_inspect[3]:.4f}]')
print()

state_t_inspect = torch.FloatTensor(state_inspect)
q_inspect = net(state_t_inspect)

print('── Output (Main Network → Q-values) ───────────────────────')
print(f'  shape  : {q_inspect.shape}   ← one value per possible action')
print(f'  Q(LEFT=0)  = {q_inspect[0].item():.4f}')
print(f'  Q(RIGHT=1) = {q_inspect[1].item():.4f}')
print(f'  → greedy action : {q_inspect.argmax().item()}  ({"LEFT" if q_inspect.argmax()==0 else "RIGHT"})')
print()
print('  Note: weights are still random — Q-values are meaningless here.')
print('        After training, high Q means "this action is rewarding from this state".')

## Step 2 · Replay Buffer
**Ch5 · Replay Buffer**

Stores raw experience tuples `(s, a, r, s′, done)` — **no Q values here**.  
Q is computed later during the TRAIN phase.

```
push(s, a, r, s′, done)  →  add to buffer
sample(32)               →  pull 32 random experiences
```

`deque(maxlen=100_000)` — when full, oldest entry is automatically overwritten.

In [ ]:
# ── Inspect B: What's inside the buffer after each push? ─────────────────────
buf_inspect = ReplayBuffer(capacity=100_000)

# 3 real-looking transitions — state evolves step by step
demo = [
    ([0.04, -0.01, -0.02,  0.03], 1, 1.0, [ 0.03,  0.18, -0.01, -0.26], False),
    ([0.03,  0.18, -0.01, -0.26], 0, 1.0, [ 0.07, -0.02, -0.04,  0.03], False),
    ([0.07, -0.02, -0.04,  0.03], 1, 0.0, [-0.01,  0.16,  0.08,  0.31], True),  # pole fell
]

print('── Pushing 3 transitions one by one ───────────────────────')
for i, (s, a, r, s2, d) in enumerate(demo):
    buf_inspect.push(s, a, r, s2, d)
    print(f'\n  push {i+1}  →  buffer size = {len(buf_inspect)}')
    print(f'    s      = {[round(v,3) for v in s]}  [x, ẋ, θ, θ̇]')
    print(f'    action = {a}  ({"LEFT" if a==0 else "RIGHT"})')
    print(f'    reward = {r}  {"← pole fell (episode ended)" if d else ""}')
    print(f'    s′     = {[round(v,3) for v in s2]}')
    print(f'    done   = {d}')

# Fill to 35 so sample(32) is possible
for _ in range(32):
    buf_inspect.push([0.0]*4, 0, 1.0, [0.01]*4, False)

print(f'\n── Filled to {len(buf_inspect)} entries — ready to sample ──────────────')
s_b, a_b, r_b, ns_b, d_b = buf_inspect.sample(32)
print(f'\n  sample(32) → batch shapes:')
print(f'    states      : {s_b.shape}   ← 32 random rows × 4 state values')
print(f'    actions     : {a_b.shape}     ← 32 integers  (0=LEFT, 1=RIGHT)')
print(f'    rewards     : {r_b.shape}     ← 32 floats    (1.0=alive, 0.0=fell)')
print(f'    next_states : {ns_b.shape}')
print(f'    dones       : {d_b.shape}')
print()
print('── First 3 sampled states ──────────────────────────────────')
for i in range(3):
    row = s_b[i].tolist()
    print(f'  [{i}]  {[round(v,4) for v in row]}')
    print(f'       x={row[0]:.4f}  ẋ={row[1]:.4f}  θ={row[2]:.4f}  θ̇={row[3]:.4f}')

In [ ]:
# ── Cell 3: Replay Buffer ──────────────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, capacity=100_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size=32):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)),
            torch.LongTensor(actions),
            torch.FloatTensor(rewards),
            torch.FloatTensor(np.array(next_states)),
            torch.FloatTensor(dones),
        )

    def __len__(self):
        return len(self.buffer)


# Quick test
buf = ReplayBuffer()
for _ in range(50):
    buf.push([0.1, 0.2, 0.3, 0.4], 0, 1.0, [0.11, 0.21, 0.31, 0.41], False)

print(f'Buffer size: {len(buf)}')
states, actions, rewards, next_states, dones = buf.sample(32)
print(f'Sampled batch shapes:')
print(f'  states      {states.shape}')
print(f'  actions     {actions.shape}')
print(f'  rewards     {rewards.shape}')
print(f'  next_states {next_states.shape}')

## Step 3 · DQN Agent
**Ch5 + Ch6** — assembling all the pieces into one class.

| Method | Role | Course connection |
|---|---|---|
| `act(state)` | ε-greedy action selection | Ch6 · exploration |
| `store(...)` | push experience to buffer | Ch5 · Replay Buffer |
| `train_step()` | sample → target Q → loss → backprop | Ch6 · Loss function |

**Target Network sync**: every `sync_every` steps, copy Main → Target weights.

In [ ]:
# ── Cell 5: Training Loop ──────────────────────────────────────────────────────
env = gym.make('CartPole-v1')

episode_rewards = []
all_losses      = []
NUM_EPISODES    = 300

# Each flag prints once, then flips True — silent for the rest of training
debug_first_step  = False
debug_first_train = False
debug_first_sync  = False

for ep in range(NUM_EPISODES):
    state, _ = env.reset()
    total_reward = 0

    while True:
        # ── PLAY ──────────────────────────────────────────────────────
        action = agent.act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # ── DEBUG A: first env step ───────────────────────────────────
        if not debug_first_step:
            with torch.no_grad():
                q_d = agent.main_net(torch.FloatTensor(state).unsqueeze(0))[0]
            print('═'*54)
            print('  STEP 1-3: Observe → ε-greedy act → env.step()')
            print('═'*54)
            print(f'\n  state   shape : (4,)')
            print(f'  state   values: {[round(v,4) for v in state]}')
            print(f'                  [x={state[0]:.4f}, ẋ={state[1]:.4f}, θ={state[2]:.4f}, θ̇={state[3]:.4f}]')
            print(f'\n  MainNet(state) → Q-values shape: {q_d.shape}')
            print(f'    Q(LEFT=0)  = {q_d[0].item():.4f}')
            print(f'    Q(RIGHT=1) = {q_d[1].item():.4f}')
            print(f'    epsilon    = {agent.epsilon:.4f}  → {"random (exploring)" if agent.epsilon > 0.5 else "greedy (exploiting)"}')
            print(f'    → action   = {action}  ({"LEFT" if action==0 else "RIGHT"})')
            print(f'\n  env.step({action}) returns:')
            print(f'    next_state : {[round(v,4) for v in next_state]}')
            print(f'    reward     : {reward}')
            print(f'    done       : {done}')
            print()
            debug_first_step = True

        # ── STORE ─────────────────────────────────────────────────────
        agent.store(state, action, reward, next_state, done)

        # ── DEBUG B: first training step ──────────────────────────────
        if not debug_first_train and len(agent.buffer) >= agent.batch_size:
            s_d, a_d, r_d, ns_d, dn_d = agent.buffer.sample(agent.batch_size)
            with torch.no_grad():
                pq = agent.main_net(s_d).gather(1, a_d.unsqueeze(1)).squeeze(1)
                mn = agent.target_net(ns_d).max(1)[0]
                tq = r_d + agent.gamma * mn * (1 - dn_d)
            print('═'*54)
            print(f'  STEPS 5-7: First training step! (buffer = {len(agent.buffer)})')
            print('═'*54)
            print(f'\n  sample({agent.batch_size}) → batch shapes:')
            print(f'    states      : {s_d.shape}   ← 32 random transitions × 4 values')
            print(f'    actions     : {a_d.shape}     ← 32 integers, e.g. {a_d[:6].tolist()} ...')
            print(f'    rewards     : {r_d.shape}     ← 32 floats,   e.g. {[round(v,1) for v in r_d[:6].tolist()]} ...')
            print(f'    next_states : {ns_d.shape}')
            print(f'    dones       : {dn_d.shape}   ← 0.0=alive, 1.0=episode ended')
            print()
            print(f'  Q-value computation — first sample in batch:')
            print(f'    state      : {[round(v,4) for v in s_d[0].tolist()]}')
            print(f'    action     : {a_d[0].item()}  ({"LEFT" if a_d[0]==0 else "RIGHT"})')
            print(f'    predicted_Q = MainNet(s)[action]       = {pq[0].item():.4f}')
            print(f'    target_Q    = r + γ × max TargetNet(s′)')
            print(f'               = {r_d[0].item():.1f} + {agent.gamma} × {mn[0].item():.4f}')
            print(f'               = {tq[0].item():.4f}')
            print(f'    loss = mean( (pred − target)² ) over {agent.batch_size} samples')
            print()
            debug_first_train = True

        # ── TRAIN ─────────────────────────────────────────────────────
        loss = agent.train_step()
        if loss is not None:
            all_losses.append(loss)

        # ── DEBUG C: first target network sync ────────────────────────
        if not debug_first_sync and agent.step_count >= agent.sync_every:
            print('═'*54)
            print(f'  STEP 8: First Target Network sync! (step {agent.step_count})')
            print('═'*54)
            print(f'  main_net weights → copied to → target_net')
            print(f'  epsilon now : {agent.epsilon:.4f}  (started at 1.0)')
            print(f'  buffer size : {len(agent.buffer)}')
            print()
            debug_first_sync = True
        # ──────────────────────────────────────────────────────────────

        state         = next_state
        total_reward += reward
        if done:
            break

    episode_rewards.append(total_reward)
    if (ep + 1) % 20 == 0:
        avg = np.mean(episode_rewards[-20:])
        bar = '█' * int(avg / 10)
        print(f'Episode {ep+1:3d} | avg reward: {avg:6.1f} {bar:<20} | ε: {agent.epsilon:.3f}')

env.close()
print('\nTraining complete!')
print(f'Last 20-episode average: {np.mean(episode_rewards[-20:]):.1f} / 200')

## Step 4 · Training Loop
**Ch6 · Full training loop** — all 8 steps running for real.

```
Every step:
  1. Observe state       →  current env state
  2. ε-greedy action     →  agent.act(state)
  3. Execute             →  env.step(action)
  4. Store               →  agent.store(...)
  5–7. Train             →  agent.train_step()
  8. Sync (automatic)    →  inside train_step(), every 1000 steps
```

In [ ]:
# ── Cell 5: Training Loop ──────────────────────────────────────────────────────
env = gym.make('CartPole-v1')

episode_rewards = []
all_losses      = []
NUM_EPISODES    = 300
debug_printed   = False   # print one detailed step, then never again

for ep in range(NUM_EPISODES):
    state, _ = env.reset()
    total_reward = 0

    while True:
        # ── PLAY ──────────────────────────────────────────────────────
        action = agent.act(state)                                        # steps 1–2
        next_state, reward, terminated, truncated, _ = env.step(action)  # step 3
        done = terminated or truncated

        # ── STORE ─────────────────────────────────────────────────────
        agent.store(state, action, reward, next_state, done)             # step 4

        # ── TRAIN ─────────────────────────────────────────────────────
        loss = agent.train_step()                                        # steps 5–8
        if loss is not None:
            all_losses.append(loss)

        # ── DEBUG: print first step only ──────────────────────────────
        if not debug_printed:
            print('=' * 52)
            print('  ONE STEP — what the data actually looks like')
            print('=' * 52)
            # CartPole state order: [x, x_dot, theta, theta_dot]
            print(f'\n  state      = {[round(v, 4) for v in state]}')
            print(f'               [x={state[0]:.4f}, ẋ={state[1]:.4f}, θ={state[2]:.4f}, θ̇={state[3]:.4f}]')
            print(f'\n  action     = {action}  ({"LEFT" if action == 0 else "RIGHT"})')
            print(f'  reward     = {reward}')
            print(f'  done       = {done}')
            print(f'\n  next_state = {[round(v, 4) for v in next_state]}')
            print(f'               [x={next_state[0]:.4f}, ẋ={next_state[1]:.4f}, θ={next_state[2]:.4f}, θ̇={next_state[3]:.4f}]')
            print(f'\n  → stored in buffer as:')
            print(f'     (s, a, r, s′, done) = ([{state[0]:.3f}...], {action}, {reward}, [{next_state[0]:.3f}...], {done})')
            print(f'\n  epsilon    = {agent.epsilon:.4f}  ({"random action" if agent.epsilon > 0.5 else "using network"})')
            print('=' * 52)
            print()
            debug_printed = True
        # ──────────────────────────────────────────────────────────────

        state         = next_state
        total_reward += reward

        if done:
            break

    episode_rewards.append(total_reward)

    if (ep + 1) % 20 == 0:
        avg = np.mean(episode_rewards[-20:])
        bar = '█' * int(avg / 10)
        print(f'Episode {ep+1:3d} | avg reward: {avg:6.1f} {bar:<20} | ε: {agent.epsilon:.3f}')

env.close()
print('\nTraining complete!')
print(f'Last 20-episode average: {np.mean(episode_rewards[-20:]):.1f} / 200')

## Step 5 · Results
**Ch6 · Atari learning curve** — the same 3-phase pattern, now on your own training run.

- **Random phase**: reward low, ε high
- **Learning phase**: reward rising, ε decaying
- **Converged**: reward ≈ 200 (CartPole max)

In [ ]:
# ── Cell 6: Results & Plots ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0f172a')
fig.suptitle('DQN CartPole — Training Results', color='white', fontsize=13, y=1.01)

for ax in [ax1, ax2]:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values():
        spine.set_color('#334155')

# ── Left: Reward curve ──
window = 20
moving_avg = [
    np.mean(episode_rewards[max(0, i - window):i + 1])
    for i in range(len(episode_rewards))
]
ax1.plot(episode_rewards, color='#22d3ee', alpha=0.25, linewidth=0.8, label='episode reward')
ax1.plot(moving_avg,      color='#22d3ee', linewidth=2.5, label=f'{window}-ep moving avg')
ax1.axhline(y=200, color='#f59e0b', linestyle='--', alpha=0.6, linewidth=1.5, label='max (200)')
ax1.set_title('Episode Reward', color='white')
ax1.set_xlabel('Episode', color='#94a3b8')
ax1.set_ylabel('Reward', color='#94a3b8')
ax1.legend(facecolor='#1e293b', labelcolor='white', fontsize=9)
ax1.set_ylim(0, 220)

# ── Right: Loss curve ──
if all_losses:
    loss_window = 200
    loss_avg = [
        np.mean(all_losses[max(0, i - loss_window):i + 1])
        for i in range(len(all_losses))
    ]
    ax2.plot(all_losses, color='#f87171', alpha=0.15, linewidth=0.5)
    ax2.plot(loss_avg,   color='#f87171', linewidth=2.5, label='moving avg')
    ax2.set_title('Training Loss  (predicted Q − target Q)²', color='white')
    ax2.set_xlabel('Training Step', color='#94a3b8')
    ax2.set_ylabel('Loss', color='#94a3b8')
    ax2.legend(facecolor='#1e293b', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('dqn_cartpole_results.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

final_avg = np.mean(episode_rewards[-20:])
if final_avg >= 195:
    print('Success! CartPole mastered (avg >= 195)')
elif final_avg >= 150:
    print(f'Learning in progress (avg {final_avg:.0f}) — try more episodes')
else:
    print(f'avg {final_avg:.0f} — try tuning hyperparameters (lr, epsilon decay, sync_every)')

## Step 6 · Watch the agent play

The physics simulation runs inside `gymnasium` — no real hardware needed.  
Every `env.step(action)` solves Newton's equations of motion for 0.02 seconds:

```
agent sends action (LEFT / RIGHT)
    ↓
gymnasium physics engine:
    F = ma   →  cart acceleration
    τ = Iα   →  pole angular acceleration
    ↓
new state [θ, θ̇, x, ẋ] returned
```

Run the cell below to render the trained agent as an animation in the notebook.

In [ ]:
# ── Cell 7: Watch the trained agent play ───────────────────────────────────────
from IPython.display import HTML
import matplotlib.animation as animation

env_render = gym.make('CartPole-v1', render_mode='rgb_array')
state, _ = env_render.reset(seed=42)
frames = []
total = 0

for _ in range(500):
    frames.append(env_render.render())
    # Use learned policy — no exploration (ε = 0)
    state_t = torch.FloatTensor(state).unsqueeze(0)
    with torch.no_grad():
        action = agent.main_net(state_t).argmax().item()
    state, reward, terminated, truncated, _ = env_render.step(action)
    total += reward
    if terminated or truncated:
        break

env_render.close()
print(f'Agent survived {int(total)} steps  (max 500)')

# Render as inline animation
fig, ax = plt.subplots(figsize=(5, 3.5))
fig.patch.set_facecolor('#0f172a')
ax.axis('off')
img = ax.imshow(frames[0])
ax.set_title(f'Trained DQN Agent — {int(total)} steps', color='white', fontsize=10)

def update(i):
    img.set_data(frames[i])
    return [img]

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=40, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

---
## What's next?

This notebook uses CartPole (state = 4 numbers).  
To scale up to **Atari** (state = 84×84×4 pixels):

```python
# Key changes: CartPole → Atari
env = gym.make('BreakoutNoFrameskip-v4')   # Atari environment
# DQNNetwork: Linear layers → Conv2d layers (Ch4 CNN pipeline)
# buffer capacity : 100,000 → 1,000,000
# NUM_EPISODES    : 300 → 10,000+
# GPU required    (DGX Spark recommended)
```

**DQN-101 Ch7** covers DQN's limits and the path forward to PPO and RLHF.